# verify07(7_20): 症状別プロトコルを1本のBERTで学習・交差検証（文ペア方式）／データ追加対応版

`dataset/7_20/` のプロトコルCSVを対象に、**1本のBERT**で「あるペア（会話の一部）が、対象ノードの質問に対してどの選択肢に当たるか」を分類する。**ファイルを追加しながら何度でも回せる**ように、保存先の自動分離とデータ変化の自動検知を入れてある。

---

## 1. データの構造とラベルの使い方 ⭐️

各CSVは **1行 = 1ペア**。列は次の通り：

| 列 | 意味 |
|---|---|
| `ID` | ペアの識別子（学習には使わない） |
| `ペア` | 会話の一部（`Dispatcher:（指令員の発話）` と応答が入ったテキスト）= **文ペアのB** |
| `is_<node>` | そのペアが**どの質問ノードに属するか**。1つだけ `True` |
| `label_<node>` | そのノードでの**正解コード**（下表） |
| その他の列（投入先モデル・意味保持度など） | 生成品質メタ。**学習には使わない** |

### ラベル（`label_<node>` = `code`）の意味
| code | 意味 | 例（2択質問なら） |
|---|---|---|
| **`0`** | **非該当** … このペアは「別の質問」に答えていて、対象ノードの質問には該当しない | — |
| **`1..N`** | そのノードの**選択肢インデックス** | `1=はい / 2=いいえ / 3=不明` |

- 選択肢数 `N` はノードによって違う（最大5。`NUM_LABELS=6` の固定ヘッド `0..5` を全ノードで共有する）。
- **なぜ `0=非該当` が要るか**：1つのBERTに全ノードを混ぜて学習するので、「このペアはそもそも今の質問の答えじゃない」を弾ける必要がある。各ノードの学習データには、**他ノードのペアが `0` として混ぜてある**。

## 2. どうやって学習するか（文ペア方式）⭐️

入力は **文ペア `(A, B)`**：
- **A = 質問文**（例「突然の激しい頭痛ですか？」）… その `node` の代表質問。**データから自動抽出**する（該当ペア `code≠0` の Dispatcher 発話の最頻文）。手直しは `NODE_Q_OVERRIDE`。
- **B = ペア本文**（`ペア` 列）

これを日本語BERT (`cl-tohoku/bert-base-japanese-v3`) に `tokenizer(A, B)` で食わせる。BERTは **segment 埋め込み**でAとBを区別し、「Bの内容はAの質問にどう答えているか」を見て `0..5` に分類する。出力ヘッドは全ノード共通の6クラス。

- 損失 = クロスエントロピー、最適化 = AdamW（`lr=2e-5`）、`3エポック`
- パディングは**バッチ内最長に動的**（`attention_mask` があるので結果は変わらず速い）
- 推論時は必ず `eval()`（dropout OFF）

つまり**「質問Aとペア本文Bを見て、Bが選択肢のどれ（or 非該当）かを当てる」**タスクを、全ノードまとめて1本のBERTに解かせている。

## 3. 交差検証（5-fold 層化）で評価 ⭐️

- データを `node×code` で**層化**して5分割（各分割で各(ノード,選択肢)の比率が揃う）
- 「4/5で学習 → 残り1/5を予測」を5回 → **全ペアが1回ずつ“学習に使われていない状態”で予測**される（= OOF予測）
- そのOOFで **全体 / 症状別 / ノード別 / code別**の正答率と混同行列を出す
- 最後に**全データで最終モデルを1本**学習して `final_model/` に保存（実運用・可視化用）

> CVは「データ全体を分割」して行うので、**あとから行を足したら分割が変わり、前回のfold結果は無効**。追記はできず、**データが変わったら全fold回し直し**が原則（下の自動化がこれを面倒なくやる）。

## 4. 「7_20 にファイルを足しながら回す」ための工夫 ⭐️

| 仕組み | 効果 |
|---|---|
| `DATA_SUBDIR` と `frac` を**保存タグに自動反映**（`checkpoints/verify07_7_20_symptom_frac2/`） | 07_19版・全件版・半分版と**絶対に混ざらない** |
| **データ指紋セル(3.5)**：`ex`＋主要設定のハッシュを保存し、前回と違えば古い `fold*_pred.csv`/`oof`/`final_model` を `_prev_<時刻>/` に**自動退避** | ファイルを足して再実行しても、**RESUMEが古い結果を誤流用しない**＝常に正しいCV |
| `EXCLUDE_PROTO_PREFIXES=['00']` | `00_common_flow`（共通フロー）は別タスクなので既定で除外（`[]` で全部入り） |

→ **運用は「7_20にCSVを足す → 上から全セル実行」だけ**。データが変わっていれば自動でCVをやり直し、変わっていなければ完了済みfoldはスキップして続きから。

## 5. 主な設定（セル4で変更）
| 項目 | 既定 | 変えたいとき |
|---|---|---|
| `DATA_FRACTION` | `1/2`（層化で半分） | 正式評価は `1.0`（全件）。もっと軽くは `1/4` 等 |
| `DATA_SUBDIR` | `7_20` | 別フォルダに変えるだけで保存先も自動分離 |
| `EXCLUDE_PROTO_PREFIXES` | `['00']` | `[]` で共通フローも1本に含める |
| `BATCH_SIZE` | `8` | GPUなら `16`/`32` で高速化 |
| `CHECKPOINT_TO_DISK` | `False` | 長時間runで切断が怖いなら `True`（毎エポック1.3GBをDriveに保存） |

**Colab GPU 推奨**（`DEVICE=cuda`）。データ量が大きいほどCVは時間がかかる（全件は数時間規模もあり得る）ので、貯めている間は `DATA_FRACTION` を下げて回すのが実用的。

# セットアップ（Colab: Drive マウント → データ取得 → pip）

データ取得は複数経路を自動で試す。このノートは **`dataset/7_20/` のプロトコルCSV**（2桁番号始まり）を対象にし、**1ファイルからでも動く**（`NEED_MIN=1`）。ファイルは今後追加していく前提。

`DATA_SOURCE='auto'` の探索順：
1. `local` … ローカル実行時にリポジトリ内の `dataset/7_20/` があればそれ
2. `drive_id` … `DRIVE_FOLDER_URL` のフォルダをID解決してCSVをDL（Colabはこれ）
3. `drive_path` … マウント済み Drive の `MyDrive/emergency_task/dataset/7_20/` 等
4. `upload` … ファイル選択ダイアログ（zip でも CSV でも可）

初回は Drive マウントと Drive API の**認証ダイアログが2回**出る。どちらも自分のアカウントを選ぶ。

**重み・ログの保存先はデータフォルダ名を含めて自動分離**（`MyDrive/emergency_task/checkpoints/verify07_7_20_symptom_frac2/` など）。07_19版や全件版と混ざらない。

In [ ]:
import os, sys, glob, io, re, subprocess, shutil, zipfile

# 共有された Drive フォルダ（この中、またはこの下に 7_21 のCSVがある想定）
DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/16y84C3Ap2rM2IQxMg2-06ko_aNylYfyS'

DATA_SOURCE = 'auto'      # 'auto' | 'local' | 'drive_id' | 'drive_path' | 'upload' | 'clone'
IN_COLAB = 'google.colab' in sys.modules

# ★このノートは dataset/7_21 を使う。ファイルは今後どんどん追加していく前提。
#   プロトコル番号2桁始まりのCSVを全部拾う（00, 02〜10, 将来の 11〜23 も自動で対象）。
DATA_SUBDIR = '7_21'
CSV_GLOB   = '[0-9][0-9]_*symptom_pair_conversations*.csv'   # 2桁プロト番号のCSV全部
CSV_RE     = r'[0-9]{2}_.*symptom_pair_conversations.*\.csv$'
NEED_MIN   = 1                                               # 1ファイルでも「フォルダ検出」OK（貯めながら回すため）
DEST_DIR   = f'/content/emergency_task/dataset/{DATA_SUBDIR}'

REPO_URL, REPO_BRANCH = 'https://github.com/enenen13/Emergency_task', 'feature/headache-ablation-notebook'


def _folder_id(url):
    m = re.search(r'/folders/([A-Za-z0-9_-]+)', url or '')
    return m.group(1) if m else None


DRIVE_FOLDER_ID = _folder_id(DRIVE_FOLDER_URL)

# ---- Drive マウント（Colabのみ）: 重み・ログの保存先 ----
DRIVE_ROOT = None
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/emergency_task'
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print('DRIVE_ROOT =', DRIVE_ROOT)

    print('pip install ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                    'transformers', 'sentencepiece', 'fugashi', 'unidic-lite',
                    'accelerate'], check=True)


def _ok(d):
    return bool(d) and os.path.isdir(d) and len(glob.glob(os.path.join(d, CSV_GLOB))) >= NEED_MIN


def _search_local():
    """カレントから上に遡って dataset/7_21 を探す。"""
    d = os.path.abspath(os.getcwd())
    for _ in range(6):
        cand = os.path.join(d, 'dataset', DATA_SUBDIR)
        if _ok(cand):
            return cand
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return None


def _drive_service():
    from google.colab import auth
    auth.authenticate_user()
    from googleapiclient.discovery import build
    return build('drive', 'v3')


def _list_children(svc, fid):
    out, tok = [], None
    while True:
        r = svc.files().list(q=f"'{fid}' in parents and trashed=false",
                             fields='nextPageToken, files(id,name,mimeType)', pageToken=tok,
                             supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        out += r.get('files', [])
        tok = r.get('nextPageToken')
        if not tok:
            return out


FOLDER_MIME = 'application/vnd.google-apps.folder'


def _collect_csvs(svc, fid, depth=0, found=None):
    """フォルダ直下だけでなくサブフォルダも3階層まで見て、目的のCSVを集める。"""
    found = {} if found is None else found
    for f in _list_children(svc, fid):
        if f['mimeType'] == FOLDER_MIME:
            if depth < 3:
                _collect_csvs(svc, f['id'], depth + 1, found)
        elif re.match(CSV_RE, f['name']):
            found.setdefault(f['name'], f['id'])
    return found


def _search_drive_id():
    """共有フォルダのIDからCSVを直接ダウンロードする（フォルダが非公開でもOK）。"""
    if not (IN_COLAB and DRIVE_FOLDER_ID):
        return None
    from googleapiclient.http import MediaIoBaseDownload
    svc = _drive_service()
    files = _collect_csvs(svc, DRIVE_FOLDER_ID)
    if len(files) < NEED_MIN:
        print(f'  [drive_id] CSVが {len(files)}/{NEED_MIN} 件しか見つからず: {sorted(files)}')
        return None
    os.makedirs(DEST_DIR, exist_ok=True)
    for name, fid in sorted(files.items()):
        dst = os.path.join(DEST_DIR, name)
        if os.path.exists(dst):
            continue
        buf = io.BytesIO()
        dl = MediaIoBaseDownload(buf, svc.files().get_media(fileId=fid, supportsAllDrives=True))
        done = False
        while not done:
            _, done = dl.next_chunk()
        with open(dst, 'wb') as fh:
            fh.write(buf.getvalue())
        print('  DL:', name, f'{len(buf.getvalue())/1e6:.1f}MB')
    return DEST_DIR if _ok(DEST_DIR) else None


def _search_drive_path():
    """マウント済み Drive の定番の場所を直接見る。"""
    if not DRIVE_ROOT:
        return None
    for cand in [os.path.join(DRIVE_ROOT, 'dataset', DATA_SUBDIR),
                 os.path.join(DRIVE_ROOT, DATA_SUBDIR),
                 os.path.join(DRIVE_ROOT, 'Emergency_task', 'dataset', DATA_SUBDIR),
                 f'/content/drive/MyDrive/{DATA_SUBDIR}']:
        if _ok(cand):
            return cand
    return None


def _do_upload():
    """zip でも CSV 直接でも受ける。"""
    from google.colab import files
    os.makedirs(DEST_DIR, exist_ok=True)
    print(f'7_21 の zip、または プロトコルCSVを選んでください（最低{NEED_MIN}ファイル）')
    up = files.upload()
    for name in up:
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(name) as z:
                z.extractall('/content/_unzip')
            os.remove(name)
            for p in glob.glob('/content/_unzip/**/' + CSV_GLOB, recursive=True):
                shutil.copy(p, DEST_DIR)
        else:
            shutil.move(name, os.path.join(DEST_DIR, os.path.basename(name)))
    return DEST_DIR if _ok(DEST_DIR) else None


def _do_clone():
    if not os.path.isdir('Emergency_task'):
        subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL], check=True)
    cand = os.path.abspath(os.path.join('Emergency_task', 'dataset', DATA_SUBDIR))
    return cand if _ok(cand) else None


FINDERS = {'local': _search_local, 'drive_id': _search_drive_id,
           'drive_path': _search_drive_path, 'upload': _do_upload, 'clone': _do_clone}
order = ['local', 'drive_id', 'drive_path', 'upload'] if DATA_SOURCE == 'auto' else [DATA_SOURCE]

QC_DIR = None
for src in order:
    try:
        QC_DIR = FINDERS[src]()
    except Exception as e:
        print(f'  [{src}] 失敗: {type(e).__name__}: {e}')
        QC_DIR = None
    if QC_DIR:
        print(f'データ検出: source={src}')
        break

assert QC_DIR, (f'7_21 のCSVが見つかりません。DRIVE_FOLDER_URL を確認するか、'
                "DATA_SOURCE='upload' にして手動アップロードしてください。")

# 出力先（Colab では Drive 固定、ローカルではリポジトリ配下）
BASE_DIR = DRIVE_ROOT if DRIVE_ROOT else os.path.abspath(os.path.join(QC_DIR, '..', '..'))
OUT_DIR = os.path.join(BASE_DIR, 'output')
os.makedirs(OUT_DIR, exist_ok=True)

print('QC_DIR  =', QC_DIR)
print('OUT_DIR =', OUT_DIR)
_found = sorted(glob.glob(os.path.join(QC_DIR, CSV_GLOB)))
print(f'見つかったCSV {len(_found)}件:')
for p in _found:
    print('  -', os.path.basename(p))

# 2. 設定`GROUP` で「症状別 / 共通」を切り替える。**いまは 02〜09 が全部症状用データなので `'symptom'`**。`CKPT_DIR` が保存先。Colab では Drive 配下（ランタイム切断で消えない）、ローカルではリポジトリ配下になる。

In [ ]:
import glob, json, random, re, time, datetime, hashlib
from typing import Dict, List

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import transformers
transformers.logging.set_verbosity_error()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# ===== このノートの識別名 =====
NB_TAG = 'verify07'   # 保存先・出力CSVの接頭辞

# ===== 学習対象グループ =====
GROUP = 'symptom'     # 'symptom' | 'common'

# グループ→対象CSV(グロブ)。7_21 にあるプロトコルCSVを全部拾う（追加ファイルも自動で対象）。
GROUP_FILES = {
    'symptom': [os.path.join(QC_DIR, '[0-9][0-9]_*.csv')],
    'common':  [],
}

# ★症状別モデルに含めないプロトコル接頭辞。00_common_flow（共通フロー）も学習に含める（トリアージ探索が共通ノード予測を要するため）。
#   すべて1本のモデルに入れたいなら [] にする。
EXCLUDE_PROTO_PREFIXES = []

# 05_convulsion 等に含まれる is_common_* ノードの扱い（症状別モデルに含めるなら False）。
EXCLUDE_COMMON_PREFIX_NODES = False

# ★データの一部だけ使う（層化サンプリング）。1.0=全件（＝全データで5分割する本番）、1/2=半分…
#   層化キーは CV と同じ node×code。各 stratum を frac 分だけ残すので分布を保ったまま縮小する。
#   ★全データが揃ったら 1.0 のまま実行 → 全ペアを 5-fold 層化で分割してCV＋各foldモデル保存。
DATA_FRACTION = 1.0

# ===== ハイパーパラメータ（verify06/05 標準）=====
BASE_MODEL      = 'cl-tohoku/bert-base-japanese-v3'   # verify05/06 と同じ
MAX_LENGTH      = 160          # 動的パディングなので長くしても遅くならない（実効長はp90=78）
LEARNING_RATE   = 2e-5         # verify06 標準
NUM_EPOCHS      = 3            # verify06 標準
BATCH_SIZE      = 8            # verify06 標準（GPUなら 16/32 に上げると速い）
EVAL_BATCH_SIZE = 32           # verify06 と同じ
N_FOLDS         = 5            # ★5分割
LEAK_SAFE_CV    = True         # ★同一ペアを同foldに固めCVリーク防止（False=従来の層化のみ）
SEED            = 42           # 分割の乱数シード（固定 → 分割は再現可能）
NUM_LABELS      = 6            # code 0..5 固定ヘッド（ノードごとに使う範囲は異なる）

SAMPLE_PER_NODE = None         # 各ノード均等N件に間引く別オプション（DATA_FRACTION とは併用可・通常 None）

# ===== チェックポイント / ログ =====
RESUME             = True      # 完了済みfoldはスキップ（下の指紋セルがデータ変化時に自動で古い結果を退避）
KEEP_FOLD_WEIGHTS  = True      # ★各foldの学習済みモデルを fold{k}_model/ に保存（1本≈450MB × 5 ≈ 2.2GB を Drive に置く）
SAVE_FINAL_MODEL   = True      # 全データ学習した最終モデルも save_pretrained する
LOG_EVERY_STEPS    = 50

# ★エポック単位のチェックポイント(重み+optimizer≈1.3GB)をディスクに書くか。
#   True  = ランタイム切断に強い（長時間runで途中エポックから再開）。ただし毎エポック1.3GB書く。
#   False = メモリ学習。ディスク/Driveをほぼ使わない。GPUなら既定 False で十分。
#   ※全データ×5fold は長いので、切断が心配なら True 推奨（fold途中エポックから再開できる）。
CHECKPOINT_TO_DISK = True 

# ★保存先タグ：データフォルダ(DATA_SUBDIR)＋frac を織り込む。
#   → 07_19版・7_21版・全件版・半分版がすべて別フォルダに自動分離され、絶対に混ざらない。
_frac_tag = 'full' if (not DATA_FRACTION or DATA_FRACTION >= 1.0) else f'frac{round(1 / DATA_FRACTION)}'
PROMPT_MODE = 'A'   # A=選択肢文なし / B=質問Aに選択肢文を付与（出力codeは同じ→verify08/09に無影響）
PROMPT_WITH_CHOICES = (PROMPT_MODE == 'B')
QUESTION_SOURCE = 'yaml'   # 'yaml'=protocol.yamlの質問を使う（推奨） / 'data'=データから最頻Dispatcher発話
RUN_TAG   = f'{DATA_SUBDIR}_{GROUP}_{_frac_tag}_{PROMPT_MODE}'                   # 例: 7_21_symptom_full
CKPT_ROOT = BASE_DIR
CKPT_DIR  = os.path.join(CKPT_ROOT, 'checkpoints', f'{NB_TAG}_{RUN_TAG}')
LOG_DIR   = os.path.join(CKPT_DIR, 'logs')
os.makedirs(LOG_DIR, exist_ok=True)

LOG_PATH = os.path.join(LOG_DIR, f'train_{datetime.datetime.now():%Y%m%d_%H%M%S}.jsonl')


def log(event: str, **kv):
    """1行ごと flush で JSONL に追記。ランタイムが落ちても記録が残る。"""
    rec = {'t': datetime.datetime.now().isoformat(timespec='seconds'), 'event': event, **kv}
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')
        f.flush()
    print('[log]', ' '.join(f'{k}={v}' for k, v in rec.items() if k != 't'))


def atomic_save(obj, path):
    """Drive 上で書き込み中に落ちても壊れないよう tmp→replace で置換。"""
    tmp = path + '.tmp'
    torch.save(obj, tmp)
    os.replace(tmp, path)


def ckpt(name):
    return os.path.join(CKPT_DIR, name)


def set_seed(seed: int = SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
print('CKPT_DIR =', CKPT_DIR)
print('LOG_PATH =', LOG_PATH)
print('DATA_FRACTION =', DATA_FRACTION, '| RUN_TAG =', RUN_TAG,
      '| KEEP_FOLD_WEIGHTS =', KEEP_FOLD_WEIGHTS, '| CHECKPOINT_TO_DISK =', CHECKPOINT_TO_DISK)
log('config', nb=NB_TAG, group=GROUP, run_tag=RUN_TAG, data_subdir=DATA_SUBDIR, data_fraction=DATA_FRACTION,
    exclude_proto=EXCLUDE_PROTO_PREFIXES, base_model=BASE_MODEL, epochs=NUM_EPOCHS, folds=N_FOLDS,
    batch=BATCH_SIZE, lr=LEARNING_RATE, max_len=MAX_LENGTH, resume=RESUME,
    keep_fold_weights=KEEP_FOLD_WEIGHTS, ckpt_to_disk=CHECKPOINT_TO_DISK)

# 3. データ読み込み：02〜09 を統合して文ペア `(質問文, ペア)` を作る各CSVは1行=1ペア。`is_<node>` が1つだけ True でそのノードに属し、`label_<node>` が `0..N` の正解。**質問文（文ペアのA）はデータから自動抽出**する：該当ペア（`label != 0`）の Dispatcher 発話のうち**最頻のもの**（同数なら短いもの）をそのノードの代表質問とする。おかしいものは `NODE_Q_OVERRIDE` で手当てする。

In [ ]:
PROTO_JP = {
    '00': '共通フロー', '02': '呼吸困難', '03': '動悸', '04': '意識障害・失神', '05': 'けいれん',
    '06': '頭痛', '07': '胸痛(非外傷)', '08': '背部痛', '09': '腰痛', '10': '成人発熱',
    '11': '腹痛', '12': '成人悪心・嘔吐', '13': 'めまい', '14': 'しびれ', '15': '吐血・喀血',
    '16': '下血・血便', '17': '倦怠感', '18': '外傷', '19': '異物誤飲', '20': '中毒',
    '21': '小児発熱', '22': '小児悪心・嘔吐', '23': '小児頭頸部外傷',
}


def _jp(pid):
    return PROTO_JP.get(pid, pid)      # 未知プロトコルでも落ちないように


# 自動抽出がいまいちなノードだけ手で上書き
NODE_Q_OVERRIDE = {
    '04_breath_count':      '呼吸と次の呼吸の間が10秒以上あきますか？',
    '05_breath_signal':     '次の呼吸まで10秒以上あきますか、10秒未満ですか？',
    '05_common_breathing':  '呼吸は普段どおりですか？',
    '09_pain_quality_65':   'どのような痛みですか？',
}


def _dispatcher_line(pair_text: str) -> str:
    m = re.search(r'Dispatcher:\s*(.*?)(?:\n|$)', str(pair_text))
    return m.group(1).strip() if m else ''


paths = sorted({p for pat in GROUP_FILES[GROUP] for p in glob.glob(pat)})
# ★除外プロトコル接頭辞を落とす（00_common_flow など）
paths = [p for p in paths if os.path.basename(p)[:2] not in EXCLUDE_PROTO_PREFIXES]
assert paths, f'GROUP={GROUP!r} に対応するCSVが見つかりません: {GROUP_FILES[GROUP]}'
print(f'対象CSV {len(paths)}件（除外接頭辞={EXCLUDE_PROTO_PREFIXES}）:')
for p in paths:
    print('  -', os.path.basename(p))

recs = []
for path in paths:
    fname = os.path.basename(path)
    pid = fname[:2]
    d = pd.read_csv(path, encoding='utf-8-sig', dtype=str)
    keys = [c[3:] for c in d.columns if c.startswith('is_')]
    for k in keys:
        if EXCLUDE_COMMON_PREFIX_NODES and k.startswith('common_'):
            continue
        on = d['is_' + k].astype(str).str.strip().str.lower().isin(['true', '1', '1.0'])  # True/False版と 1.0/空欄版の両対応
        sub = d.loc[on, ['ID', 'ペア', 'label_' + k]]
        for _, r in sub.iterrows():
            recs.append({'proto': pid, 'node': f'{pid}_{k}', 'row_id': r['ID'],
                         'text': str(r['ペア']), 'code': int(float(str(r['label_' + k]).strip()))})

ex = pd.DataFrame(recs)
assert len(ex), '行が0件です'

# ---- ノードごとの代表質問（文ペアのA）を自動抽出 ----
NODE_Q: Dict[str, str] = {}
for node, g in ex[ex['code'] != 0].groupby('node'):
    cnt = g['text'].map(_dispatcher_line).value_counts()
    # 最頻 → 同数なら短い方
    best = sorted(cnt.items(), key=lambda kv: (-kv[1], len(kv[0])))[0][0]
    NODE_Q[node] = NODE_Q_OVERRIDE.get(node, best)
for k, v in NODE_Q_OVERRIDE.items():
    if k in NODE_Q:
        NODE_Q[k] = v

# ★質問A(prompt)の作り方: QUESTION_SOURCE('yaml'=yamlの質問 / 'data'=データ抽出) ＋ 選択肢付与(案B)
if QUESTION_SOURCE == 'yaml' or PROMPT_WITH_CHOICES:
    import yaml as _yaml
    def _find_or_fetch(rel):
        _d = os.path.abspath(os.getcwd())
        for _ in range(6):
            _c = os.path.join(_d, rel)
            if os.path.exists(_c):
                return _c
            _d = os.path.dirname(_d)
        if not os.path.isdir('Emergency_task'):
            subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL], check=True)
        _c = os.path.join('Emergency_task', rel)
        return _c if os.path.exists(_c) else None
    _yp = _find_or_fetch('transition_diagram/protocol.yaml')
    _nmp = _find_or_fetch('transition_diagram/node_map.json')
    assert _yp and _nmp, 'protocol.yaml / node_map.json が必要です'
    _proto = _yaml.safe_load(open(_yp, encoding='utf-8'))
    _yq, _ych = {}, {}
    def _collect(_n):
        if 'question' in _n:
            _yq[_n['id']] = re.sub(r'^\s*[（(].*?[）)]\s*', '', str(_n['question']))
        if 'choices' in _n:
            _ych[_n['id']] = [str(c.get('text', '')) for c in _n['choices']]
    for _sec in ('entry_flow', 'common_vitals'):
        for _n in _proto.get(_sec, []):
            _collect(_n)
    for _p in _proto.get('protocols', []):
        for _n in _p.get('nodes', []):
            _collect(_n)
    _nm = json.load(open(_nmp, encoding='utf-8'))
    _bq = {b: _yq[y] for y, b in _nm.items() if y in _yq}
    _bch = {}
    for _y, _b in _nm.items():
        if _y in _ych:
            _bch.setdefault(_b, _ych[_y])
    if QUESTION_SOURCE == 'yaml':
        _n1 = 0
        for _node in list(NODE_Q):
            if _node in _bq and _bq[_node].strip():
                NODE_Q[_node] = _bq[_node]; _n1 += 1
        print(f'[QUESTION_SOURCE=yaml] yaml質問で上書き: {_n1} / {len(NODE_Q)} ノード（残りはデータ抽出）')
    if PROMPT_WITH_CHOICES:
        _n2 = 0
        for _node in list(NODE_Q):
            _ch = _bch.get(_node)
            if _ch:
                _enum = ' / '.join([f'{_i+1}){_t}' for _i, _t in enumerate(_ch)])
                NODE_Q[_node] = f'{NODE_Q[_node]}（選択肢: {_enum}）'; _n2 += 1
        print(f'[案B] 選択肢文を付与: {_n2} ノード')

# ★A用の検証済辞書（transition_diagram/node_questions_A.json）があれば、Aの質問はそれを最優先で使う。
#   Colabでも常に同じ・検証済の質問Aになる（common_* のpid差も吸収済）。
try:
    _ff = _find_or_fetch
except NameError:
    _ff = None
if PROMPT_MODE == 'A' and _ff is not None:
    _nqa = _ff('transition_diagram/node_questions_A.json')
    if _nqa and os.path.exists(_nqa):
        _qd = json.load(open(_nqa, encoding='utf-8')).get('questions', {})
        _nA = 0
        for _node in list(NODE_Q):
            _qv = _qd.get(_node, {}).get('question')
            if _qv:
                NODE_Q[_node] = _qv; _nA += 1
        print(f'[node_questions_A.json] 検証済A質問で上書き: {_nA}/{len(NODE_Q)} ノード')

ex['prompt'] = ex['node'].map(NODE_Q)
assert ex['prompt'].notna().all(), '質問文を抽出できないノードがあります'

if SAMPLE_PER_NODE:
    _n = int(min(SAMPLE_PER_NODE, ex.groupby('node').size().min()))
    ex = ex.groupby('node', group_keys=False).sample(n=_n, random_state=SEED)
    print(f'[SAMPLE_PER_NODE={SAMPLE_PER_NODE}] 各ノード {_n} 件に間引き')

# ★層化サンプリング：node×code を stratum として各群を DATA_FRACTION 分だけ残す。
#   分布を保ったまま全体を縮小する。CV も同じ node×code で層化するので整合する。
if DATA_FRACTION and DATA_FRACTION < 1.0:
    _n_before = len(ex)
    _key = ex['node'].astype(str) + '_' + ex['code'].astype(str)
    ex = (ex.groupby(_key, group_keys=False)
            .sample(frac=DATA_FRACTION, random_state=SEED))
    _min_strat = ex.groupby(['node', 'code']).size().min()
    print(f'[DATA_FRACTION={DATA_FRACTION}] node×code 層化で {_n_before}→{len(ex)} 件に間引き '
          f'（最小 stratum={_min_strat}）')
    assert _min_strat >= N_FOLDS, (
        f'層化サンプリング後、最小 stratum={_min_strat} < N_FOLDS={N_FOLDS} のため 5-fold 層化が組めません。'
        f' DATA_FRACTION を上げるか N_FOLDS を下げてください。')

ex = ex.reset_index(drop=True)
ex['label'] = ex['code']            # code をそのままラベルIDに（0..5）
assert ex['label'].between(0, NUM_LABELS - 1).all(), f'label が 0..{NUM_LABELS-1} を外れています'

print(f'\nペア数: {len(ex)} / プロトコル: {ex["proto"].nunique()} / ノード: {ex["node"].nunique()} '
      f'/ ラベル値: {sorted(ex["code"].unique())}')
log('dataset', pairs=len(ex), nodes=int(ex['node'].nunique()), protos=int(ex['proto'].nunique()))

# ノード一覧（代表質問と選択肢数）
node_info = (ex.groupby(['proto', 'node'])
               .agg(n=('code', 'size'),
                    n_choices=('code', lambda s: int(s[s != 0].max())),
                    n_nonapplicable=('code', lambda s: int((s == 0).sum())))
               .reset_index())
node_info.insert(1, '症状', node_info['proto'].map(_jp))
node_info['質問文(自動抽出)'] = node_info['node'].map(NODE_Q)
display(node_info)
node_info.to_csv(os.path.join(CKPT_DIR, 'node_info.csv'), index=False, encoding='utf-8-sig')
print('saved:', os.path.join(CKPT_DIR, 'node_info.csv'))

print('\nプロトコル×code 件数:')
display(pd.crosstab(ex['proto'].map(lambda p: f'{p} {_jp(p)}'), ex['code']))
display(ex[['proto', 'node', 'prompt', 'text', 'code']].head(3))

In [ ]:
# ===== 3.5 データ変化の自動検知 → 古いCV結果を自動退避（RESUME取り違え防止）=====
# ファイルを足して回すと、CVの分割が変わるので前回の fold*_pred.csv / fold*_model は無効。
# ここで「今の ex（＝実際に学習に使うデータ）＋主要設定」の指紋を取り、前回と違えば
# 古い fold*_pred.csv / fold*_model / oof / final_model / 分割記録 を _prev_<時刻>/ に退避してやり直す。
# → あなたは何も気にせず「ファイル足す → 上から実行」だけで常に正しいCVになる。

def _dataset_fingerprint():
    ids = sorted((ex['row_id'].astype(str) + ':' + ex['code'].astype(str)).tolist())
    cfg = [DATA_FRACTION, SEED, N_FOLDS, BASE_MODEL, MAX_LENGTH, NUM_LABELS,
           LEARNING_RATE, NUM_EPOCHS, BATCH_SIZE, sorted(EXCLUDE_PROTO_PREFIXES)]
    payload = '|'.join(ids) + '||' + repr(cfg) + '||' + '|'.join(sorted(ex['node'].unique()))
    return hashlib.md5(payload.encode('utf-8')).hexdigest()

_FP_PATH = ckpt('data_fingerprint.txt')
_fp_now  = _dataset_fingerprint()
_fp_prev = open(_FP_PATH, encoding='utf-8').read().strip() if os.path.exists(_FP_PATH) else None

if _fp_prev == _fp_now:
    print(f'[fingerprint] 前回と同一データ/設定（{_fp_now[:12]}…）→ 完了済みfoldはRESUMEでスキップ。')
elif _fp_prev is None:
    print(f'[fingerprint] 初回（{_fp_now[:12]}…）。指紋を記録して通常実行。')
else:
    _stale = (glob.glob(ckpt('fold*_pred.csv')) + glob.glob(ckpt('fold*_model')) +
              glob.glob(ckpt('*_ckpt.pt')) +
              [ckpt('oof_all.csv'), ckpt('final_model'), ckpt('fold_assignments.csv')])
    _stale = [p for p in _stale if os.path.exists(p)]
    if _stale:
        _arch = ckpt('_prev_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
        os.makedirs(_arch, exist_ok=True)
        for _p in _stale:
            shutil.move(_p, os.path.join(_arch, os.path.basename(_p)))
        print(f'[fingerprint] ★データ/設定が前回と変化 → 古い結果を {os.path.basename(_arch)}/ に退避しCVやり直し。')
        print('   退避:', [os.path.basename(p) for p in _stale])
        log('data_changed', arch=os.path.basename(_arch), moved=len(_stale))
    else:
        print('[fingerprint] ★データ/設定が変化（退避対象なし）→ 新規にCV。')

with open(_FP_PATH, 'w', encoding='utf-8') as _f:
    _f.write(_fp_now)

# 4. 学習・推論部品（文ペア方式・verify05 と同じ）`tokenizer(prompt, text)` で A=質問文 / B=ペア本文 を segment 埋め込みで区別させる。推論前に `eval()`（dropout OFF）を徹底。

In [ ]:
_tok_cache: Dict[str, 'AutoTokenizer'] = {}


def get_tokenizer(name):
    if name not in _tok_cache:
        _tok_cache[name] = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    return _tok_cache[name]


class PairDataset(Dataset):
    """文ペア (prompt=A, text=B) → ラベル。トークン化は collate 側（動的パディング）。"""

    def __init__(self, prompts, texts, labels):
        self.prompts, self.texts, self.labels = list(prompts), list(texts), list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.prompts[idx], self.texts[idx], self.labels[idx]


def make_collate(tok):
    """バッチ内の最長に合わせてパディングする。
    verify05 は padding='max_length' で全部128に膨らませていたが、実効長は p90=78。
    attention_mask があるのでパディング量は結果に影響せず、計算量だけが減る（≒2倍速）。"""
    def collate(batch):
        ps, ts, ls = zip(*batch)
        enc = tok(list(ps), list(ts), truncation=True, max_length=MAX_LENGTH,
                  padding=True, return_tensors='pt')
        enc['labels'] = torch.tensor(ls, dtype=torch.long)
        return enc
    return collate


def build_model(name=BASE_MODEL):
    model = AutoModelForSequenceClassification.from_pretrained(
        name, num_labels=NUM_LABELS, trust_remote_code=True)
    tok = get_tokenizer(name)
    if len(tok) != model.config.vocab_size:
        print(f'  [警告] 語彙数{len(tok)}!=vocab{model.config.vocab_size} → resize')
        model.resize_token_embeddings(len(tok))
    return model.to(DEVICE)


@torch.no_grad()
def predict_pairs(model, tok, prompts, texts):
    model.eval()
    prompts, texts = list(prompts), list(texts)
    preds = []
    for i in range(0, len(texts), EVAL_BATCH_SIZE):
        enc = tok(prompts[i:i + EVAL_BATCH_SIZE], texts[i:i + EVAL_BATCH_SIZE],
                  truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors='pt')
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        logits = model(**enc).logits
        preds.extend(logits.argmax(-1).cpu().tolist())
    return preds


def free_memory(*objs):
    for o in objs:
        del o
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 5. エポック単位チェックポイント付き学習ループ`train_with_checkpoint(tag, prompts, texts, labels)` は1. `CKPT_DIR/<tag>_ckpt.pt` があれば **重み+optimizer+完了エポック数** を読んで**続きから**学習2. 1エポック終わるたびに `<tag>_ckpt.pt` を**アトミック置換**で上書き（常に1ファイル ≈ 450MB）3. `LOG_EVERY_STEPS` ごとに loss をログへ flushDataLoader の shuffle は `set_seed(SEED + epoch)` でエポックごとに固定するので、再開しても「そのエポックの並び」は初回実行と同じになる。

In [ ]:
def train_with_checkpoint(tag: str, prompts, texts, labels, num_epochs=NUM_EPOCHS):
    """tag ごとに <tag>_ckpt.pt を持ち、エポック単位でレジュームする。
    CHECKPOINT_TO_DISK=False のときは一切ディスクに書かず、メモリ上だけで学習する
    （ディスク/Drive を消費しない。落ちたら最初からやり直し）。"""
    ck_path = ckpt(f'{tag}_ckpt.pt')
    tok = get_tokenizer(BASE_MODEL)
    set_seed(SEED)
    model = build_model(BASE_MODEL)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    start_epoch = 0
    if RESUME and CHECKPOINT_TO_DISK and os.path.exists(ck_path):
        state = torch.load(ck_path, map_location=DEVICE)
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optim'])
        start_epoch = state['epoch'] + 1
        log('resume', tag=tag, from_epoch=start_epoch, prev_loss=round(state.get('loss', -1), 4))
        if start_epoch >= num_epochs:
            model.eval()
            return model, tok

    ds = PairDataset(prompts, texts, labels)
    collate = make_collate(tok)
    for epoch in range(start_epoch, num_epochs):
        set_seed(SEED + epoch)                     # エポックごとに並びを決定的に
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
        model.train()
        t0, run_loss, nstep = time.time(), 0.0, 0
        for step, batch in enumerate(loader, 1):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(**batch)
            out.loss.backward()
            optimizer.step()
            run_loss += float(out.loss); nstep += 1
            if step % LOG_EVERY_STEPS == 0:
                log('step', tag=tag, epoch=epoch, step=f'{step}/{len(loader)}',
                    loss=round(run_loss / nstep, 4), sec=round(time.time() - t0))
                run_loss, nstep = 0.0, 0
        epoch_loss = float(out.loss)
        if CHECKPOINT_TO_DISK:                     # ★重み+optimizer(≈1.3GB)を書くのはここだけ
            atomic_save({'model': model.state_dict(), 'optim': optimizer.state_dict(),
                         'epoch': epoch, 'loss': epoch_loss, 'tag': tag}, ck_path)
        log('epoch_done', tag=tag, epoch=epoch, loss=round(epoch_loss, 4),
            sec=round(time.time() - t0), ckpt=(os.path.basename(ck_path) if CHECKPOINT_TO_DISK else 'mem'))

    model.eval()      # 推論は必ず eval（dropout OFF）
    return model, tok

# 6. 5-fold CV（node×code 層化）：全ペアのOOF予測 ＋ 分割の記録 ＋ 各foldモデル保存

- **分割方法**：`node×code` で層化して `SEED=42` 固定の5分割（同じデータなら毎回同一・再現可能）。
  どのペアがどの fold の検証(test)に入るかを **`fold_assignments.csv`** に保存する。
- **各foldのモデル**：`KEEP_FOLD_WEIGHTS=True` なので、fold ごとに「4/5で学習した重み」を
  **`fold{k}_model/`** に `save_pretrained`（`node_questions.json` も同梱＝そのfold単体で推論可）。
- **fold単位のレジューム**：`fold{k}_pred.csv` があればその fold はスキップ。ランタイムが切れても
  このセルを再実行するだけで続きから（`CHECKPOINT_TO_DISK=True` なら fold 途中エポックからも再開）。
- 全fold終了で `oof_all.csv`（全ペアのOOF予測）を作る。

> 5モデル×約450MB ≈ 2.2GB ＋ 最終モデル約450MB を Drive に保存する。容量に注意。
> fold重みは要らず正答率だけ見たいなら `KEEP_FOLD_WEIGHTS=False` にする。

In [ ]:
# ===== 5-fold 層化分割（node×code）=====
strat = (ex['node'].astype(str) + '_' + ex['code'].astype(str)).values
if LEAK_SAFE_CV:
    # ★リーク防止：同一ペア(本文)を同じ fold に固める（node×code 層化 × ペアでグループ）
    from sklearn.model_selection import StratifiedGroupKFold
    _groups = ex['text'].values
    _sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    splits = list(_sgkf.split(ex.index, strat, _groups))
    _leak = sum(len(set(ex.loc[tr,'text']) & set(ex.loc[te,'text'])) for tr,te in splits)
    print(f'[LEAK_SAFE_CV] 同一ペアの train/test 跨ぎ = {_leak}（0でリークなし）')
else:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    splits = list(skf.split(ex.index, strat))

# --- 分割方法の記録：各ペアが「どのfoldの検証(test)に入るか」をCSVに保存 ---
#   同じデータ・同じ SEED なら毎回この分割は同一（再現可能）。
fold_of = np.empty(len(ex), dtype=int)
for k, (tr, te) in enumerate(splits):
    fold_of[te] = k
split_df = ex[['row_id', 'proto', 'node', 'code']].copy()
split_df['fold'] = fold_of                          # そのペアが test に入る fold 番号
split_df.to_csv(ckpt('fold_assignments.csv'), index=False, encoding='utf-8-sig')
print('分割を保存 →', ckpt('fold_assignments.csv'))
print('fold別 件数(test):', {int(k): int((fold_of == k).sum()) for k in range(N_FOLDS)})
log('split_saved', n=len(ex), folds=N_FOLDS, seed=SEED, per_fold_test=[int((fold_of == k).sum()) for k in range(N_FOLDS)])

fold_frames = []
for fold, (tr, te) in enumerate(splits):
    pred_path = ckpt(f'fold{fold}_pred.csv')
    if RESUME and os.path.exists(pred_path):
        fold_frames.append(pd.read_csv(pred_path))
        log('fold_skip', fold=fold, reason='pred.csv あり（完了済み）')
        continue

    t0 = time.time()
    log('fold_start', fold=fold, train=len(tr), test=len(te))
    model, tok = train_with_checkpoint(f'fold{fold}',
                                       ex.loc[tr, 'prompt'].tolist(),
                                       ex.loc[tr, 'text'].tolist(),
                                       ex.loc[tr, 'label'].tolist())
    preds = predict_pairs(model, tok, ex.loc[te, 'prompt'].tolist(), ex.loc[te, 'text'].tolist())

    fdf = ex.loc[te, ['row_id', 'proto', 'node', 'code', 'label']].copy()
    fdf['pred'] = preds
    fdf['fold'] = fold
    fdf.to_csv(pred_path, index=False, encoding='utf-8-sig')   # ← fold の成果はここで確定
    fold_frames.append(fdf)

    acc = float((fdf['pred'] == fdf['label']).mean())
    log('fold_done', fold=fold, acc=round(acc, 4), sec=round(time.time() - t0), saved=os.path.basename(pred_path))

    # --- 各foldの学習済みモデルを保存（KEEP_FOLD_WEIGHTS=True）---
    if KEEP_FOLD_WEIGHTS:
        fdir = ckpt(f'fold{fold}_model')
        model.save_pretrained(fdir)
        tok.save_pretrained(fdir)
        # そのfoldモデル単体で (質問文, ペア) を推論できるよう、ノード→質問文も同梱
        with open(os.path.join(fdir, 'node_questions.json'), 'w', encoding='utf-8') as f:
            json.dump({'group': GROUP, 'num_labels': NUM_LABELS, 'fold': fold,
                       'train_n': int(len(tr)), 'test_n': int(len(te)),
                       'node_questions': NODE_Q}, f, ensure_ascii=False, indent=2)
        log('fold_model_saved', fold=fold, dir=os.path.basename(fdir))
    else:
        if os.path.exists(ckpt(f'fold{fold}_ckpt.pt')):
            os.remove(ckpt(f'fold{fold}_ckpt.pt'))    # 完了したので途中重みは破棄（Drive節約）
    free_memory(model)

oof = pd.concat(fold_frames, ignore_index=True)
oof.to_csv(ckpt('oof_all.csv'), index=False, encoding='utf-8-sig')
assert len(oof) == len(ex), f'OOF件数不一致: {len(oof)} != {len(ex)}'
log('cv_done', n=len(oof), acc=round(float((oof['pred'] == oof['label']).mean()), 4))
print('OOF予測 完了 →', ckpt('oof_all.csv'))

# 7. 結果：全体 / 症状別 / ノード別 / ラベル別の正答率

In [ ]:
oof['correct'] = (oof['pred'] == oof['label']).astype(int)
overall_acc = oof['correct'].mean()
overall_f1 = f1_score(oof['label'], oof['pred'], average='macro', zero_division=0)
print(f'=== 全体 ===  accuracy={overall_acc:.3f}  macro-F1={overall_f1:.3f}  (n={len(oof)})')

# --- 症状（プロトコル）別 ---
proto_tbl = (oof.groupby('proto').agg(n=('correct', 'size'), 正答率=('correct', 'mean')).reset_index())
proto_tbl.insert(1, '症状', proto_tbl['proto'].map(PROTO_JP))
print('\n===== 症状（プロトコル）別 正答率 =====')
display(proto_tbl.assign(正答率=lambda t: t['正答率'].map('{:.3f}'.format)))

# --- ノード別 ---
node_tbl = (oof.groupby(['proto', 'node']).agg(n=('correct', 'size'), 正答率=('correct', 'mean')).reset_index())
node_tbl.insert(1, '症状', node_tbl['proto'].map(PROTO_JP))
node_tbl['質問文'] = node_tbl['node'].map(NODE_Q)
print('\n===== 質問ノード別 正答率 =====')
display(node_tbl.sort_values('正答率').assign(正答率=lambda t: t['正答率'].map('{:.3f}'.format)))

# --- 非該当(0) vs 該当(1..N) ---
oof['種別'] = np.where(oof['code'] == 0, '非該当(0)', '該当(1..N)')
print('\n===== 非該当 / 該当 別 正答率 =====')
display(oof.groupby('種別').agg(n=('correct', 'size'), 正答率=('correct', 'mean'))
           .reset_index().assign(正答率=lambda t: t['正答率'].map('{:.3f}'.format)))

# --- code 別 ---
print('\n===== code 別 正答率 =====')
display(oof.groupby('code').agg(n=('correct', 'size'), 正答率=('correct', 'mean'))
           .reset_index().assign(正答率=lambda t: t['正答率'].map('{:.3f}'.format)))

print('\n===== 混同行列（正解 code × 予測 code）=====')
display(pd.crosstab(oof['code'], oof['pred'], rownames=['正解'], colnames=['予測']))

for name, tbl in [('proto_acc', proto_tbl), ('node_acc', node_tbl)]:
    for d in {OUT_DIR, CKPT_DIR}:
        tbl.to_csv(os.path.join(d, f'{NB_TAG}_{RUN_TAG}_{name}.csv'), index=False, encoding='utf-8-sig')
print(f'\nsaved: {NB_TAG}_{RUN_TAG}_proto_acc.csv, {NB_TAG}_{RUN_TAG}_node_acc.csv （output/ と CKPT_DIR の両方）')

# 8. 全データで最終モデルを学習して保存（任意）CV は評価用なので、実運用に使う重みは**全ペアで1本**学習して `final_model/` に `save_pretrained` する。ここもエポック単位レジューム付き。`SAVE_FINAL_MODEL=False` なら丸ごとスキップ。

In [ ]:
FINAL_DIR = ckpt('final_model')

if not SAVE_FINAL_MODEL:
    print('SAVE_FINAL_MODEL=False → スキップ')
elif RESUME and os.path.exists(os.path.join(FINAL_DIR, 'config.json')):
    print('最終モデルは保存済み →', FINAL_DIR)
else:
    t0 = time.time()
    log('final_start', n=len(ex))
    final_model, final_tok = train_with_checkpoint('final', ex['prompt'].tolist(),
                                                   ex['text'].tolist(), ex['label'].tolist())
    os.makedirs(FINAL_DIR, exist_ok=True)
    final_model.save_pretrained(FINAL_DIR)
    final_tok.save_pretrained(FINAL_DIR)
    with open(os.path.join(FINAL_DIR, 'node_questions.json'), 'w', encoding='utf-8') as f:
        json.dump({'group': GROUP, 'num_labels': NUM_LABELS, 'node_questions': NODE_Q}, f,
                  ensure_ascii=False, indent=2)
    log('final_done', dir=FINAL_DIR, sec=round(time.time() - t0))
    print('最終モデル保存 →', FINAL_DIR)
    if not KEEP_FOLD_WEIGHTS and os.path.exists(ckpt('final_ckpt.pt')):
        os.remove(ckpt('final_ckpt.pt'))
    free_memory(final_model)

print('\n=== CKPT_DIR の中身 ===')
for f in sorted(os.listdir(CKPT_DIR)):
    p = os.path.join(CKPT_DIR, f)
    size = sum(os.path.getsize(os.path.join(p, x)) for x in os.listdir(p)) if os.path.isdir(p) else os.path.getsize(p)
    print(f'  {f:28s} {size/1e6:8.1f} MB')

# 9. 保存した最終モデルで推論する（使い方サンプル）学習セルを回さなくても、`final_model/` から読み直して `(質問文, ペア)` を判定できる。

In [ ]:
if os.path.exists(os.path.join(FINAL_DIR, 'config.json')):
    inf_tok = AutoTokenizer.from_pretrained(FINAL_DIR)
    inf_model = AutoModelForSequenceClassification.from_pretrained(FINAL_DIR).to(DEVICE).eval()
    meta = json.load(open(os.path.join(FINAL_DIR, 'node_questions.json'), encoding='utf-8'))
    NODE_Q_LOADED = meta['node_questions']

    @torch.no_grad()
    def classify(node_key: str, pair_text: str):
        q = NODE_Q_LOADED[node_key]
        enc = inf_tok(q, pair_text, truncation=True, max_length=MAX_LENGTH, return_tensors='pt').to(DEVICE)
        prob = inf_model(**enc).logits.softmax(-1)[0].cpu().numpy()
        code = int(prob.argmax())
        return code, float(prob[code]), prob

    for _, r in ex.groupby('proto', group_keys=False).head(1).iterrows():
        c, p, _ = classify(r['node'], r['text'])
        mark = 'OK' if c == r['code'] else 'NG'
        print(f'[{mark}] {r["node"]:28s} 正解code={r["code"]} 予測code={c} (conf={p:.2f})')
        print('     Q:', NODE_Q_LOADED[r['node']])
        print('     P:', r['text'].replace('\n', ' | ')[:80], '\n')
else:
    print('final_model が無いのでスキップ（セル8を先に実行）')

# 9.5 Integrated Gradients (IG)：予測根拠のトークン寄与度attention は「どこを見たか」しか分からない（符号なし・層依存）。**IG は「予測クラスの logit をどのトークンがどれだけ押し上げた/引き下げたか」を符号付きで**返すので、根拠の説明として素直。- 対象は `final_model/`（全データ学習した最終モデル）。入力は verify05/06 と同じ **文ペア `(質問文, ペア)`**- **層積分IG**（`LayerIntegratedGradients`）を **単語埋め込み層**に対して適用。ベースラインは `[PAD]`（特殊トークンは保持）- ターゲットは**予測クラス**（argmax）。赤=予測を押し上げた寄与 / 青=引き下げた寄与- サンプルは **`IG_N=10` を症状(proto)で層化抽出**（8症状にばらける）- `convergence delta`（積分誤差）も出す。小さいほど IG の近似が信頼できるColab では `captum` が要るので、無ければ自動 pip する。attention セルより**前**に置いてある。

In [ ]:
IG_N       = 10      # 可視化するサンプル数（症状で層化）
IG_NSTEPS  = 50      # 積分ステップ数（多いほど正確・遅い）
IG_LAYERS  = -1      # （未使用・将来用）

if not os.path.exists(os.path.join(FINAL_DIR, 'config.json')):
    print('final_model が無いのでスキップ（セル8を先に実行）')
else:
    try:
        from captum.attr import LayerIntegratedGradients
    except ImportError:
        print('captum を入れます...'); subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'captum'], check=True)
        from captum.attr import LayerIntegratedGradients
    from IPython.display import display, HTML

    ig_tok = AutoTokenizer.from_pretrained(FINAL_DIR)
    ig_model = AutoModelForSequenceClassification.from_pretrained(FINAL_DIR).to(DEVICE).eval()
    ig_meta = json.load(open(os.path.join(FINAL_DIR, 'node_questions.json'), encoding='utf-8'))
    IG_NODE_Q = ig_meta['node_questions']
    emb_layer = ig_model.get_input_embeddings()      # 単語埋め込み層（この出力に対して積分）

    def _ig_forward(input_ids, attention_mask, token_type_ids):
        return ig_model(input_ids=input_ids, attention_mask=attention_mask,
                        token_type_ids=token_type_ids).logits

    lig = LayerIntegratedGradients(_ig_forward, emb_layer)

    @torch.no_grad()
    def _predict_code(enc):
        return int(ig_model(**enc).logits.argmax(-1).cpu())

    def ig_explain(node_key, pair_text, target=None):
        q = IG_NODE_Q[node_key]
        enc = ig_tok(q, pair_text, truncation=True, max_length=MAX_LENGTH,
                     return_tensors='pt', return_token_type_ids=True).to(DEVICE)
        input_ids, attn = enc['input_ids'], enc['attention_mask']
        # token_type_ids を返さないトークナイザ設定でもゼロ埋めで対応
        ttype = enc['token_type_ids'] if 'token_type_ids' in enc else torch.zeros_like(input_ids)
        pred = _predict_code(enc) if target is None else target

        # ベースライン: 特殊トークン以外を [PAD] に。特殊トークン位置は保持。
        special = torch.tensor(ig_tok.get_special_tokens_mask(input_ids[0].tolist(),
                               already_has_special_tokens=True), device=DEVICE).bool()
        ref = torch.where(special, input_ids[0], torch.full_like(input_ids[0], ig_tok.pad_token_id)).unsqueeze(0)

        atts, delta = lig.attribute(inputs=input_ids, baselines=ref,
                                    additional_forward_args=(attn, ttype), target=pred,
                                    n_steps=IG_NSTEPS, internal_batch_size=8,
                                    return_convergence_delta=True)
        scores = atts.sum(dim=-1).squeeze(0)                 # 埋め込み次元を集約
        scores = scores / (scores.abs().max() + 1e-9)        # [-1,1] に正規化
        tokens = ig_tok.convert_ids_to_tokens(input_ids[0])
        return tokens, scores.detach().cpu().tolist(), ttype[0].cpu().tolist(), pred, float(delta[0])

    def _render_ig(node_key, pair_text, tokens, scores, ttype, pred, delta):
        spans = []
        for t, s, seg in zip(tokens, scores, ttype):
            if t in ('[PAD]',):
                continue
            # 赤=正の寄与 / 青=負の寄与。|s| を濃さに
            r, g, b = (255, 60, 60) if s >= 0 else (60, 90, 255)
            a = min(abs(s), 1.0)
            border = 'border-bottom:2px solid #999;' if seg == 1 else ''   # segment B(ペア)に下線
            spans.append(f'<span style="background-color: rgba({r},{g},{b},{a:.3f}); padding:2px; '
                         f'margin:1px; border-radius:3px; {border}">{t}</span>')
        display(HTML(f'<b>{node_key} / 予測code={pred}</b>'
                     f'<span style="color:#888"> (Δ={delta:+.3f})</span><br>'
                     f'<span style="color:#888">A(質問)={node_key and IG_NODE_Q[node_key]}</span><br>'
                     + ' '.join(spans)))

    # --- IG_N を症状(proto)で層化抽出（ラウンドロビン: IG_N<症状数 でも壊れない）---
    def _stratified_sample(df, col, n, seed):
        pools = {k: g.sample(frac=1, random_state=seed).index.tolist()
                 for k, g in df.groupby(col)}
        order, picks = sorted(pools), []
        while len(picks) < n and any(pools[k] for k in order):
            for k in order:
                if pools[k]:
                    picks.append(pools[k].pop())
                    if len(picks) >= n:
                        break
        return df.loc[picks]

    ig_sample = (_stratified_sample(ex, 'proto', min(IG_N, len(ex)), SEED)
                 .sort_values('proto').reset_index(drop=True))
    print(f'IG対象 {len(ig_sample)}件（症状で層化）:', ig_sample['proto'].map(PROTO_JP).value_counts().to_dict())
    display(HTML('<div style="color:#888">凡例: '
                 '<span style="background:rgba(255,60,60,.7);padding:2px">赤=予測を押し上げ</span> '
                 '<span style="background:rgba(60,90,255,.7);padding:2px">青=引き下げ</span> '
                 '／ 下線 = segment B(ペア本文)</div>'))

    for _, r in ig_sample.iterrows():
        toks, sc, tt, pred, delta = ig_explain(r['node'], r['text'])
        mark = 'OK' if pred == r['code'] else 'NG'
        print(f'[{mark}] node={r["node"]} 正解code={r["code"]}')
        print('   P:', r['text'].replace(chr(10), ' | ')[:90])
        _render_ig(r['node'], r['text'], toks, sc, tt, pred, delta)

# 10. Attention 可視化（任意・重いのでスキップ可）`final_model/` の重みを **attn_implementation='eager'** で読み直し、`(質問文, ペア)` の CLS→各トークンのattention（最終層・ヘッド平均）を色付け表示する。「はい/いいえ」だけでなく**内容語**を見ているか確認する用。

In [ ]:
RUN_ATTENTION_VIZ = True

if RUN_ATTENTION_VIZ and os.path.exists(os.path.join(FINAL_DIR, 'config.json')):
    from IPython.display import display, HTML

    viz_tok = AutoTokenizer.from_pretrained(FINAL_DIR)
    viz_model = AutoModelForSequenceClassification.from_pretrained(
        FINAL_DIR, attn_implementation='eager', output_attentions=True).to(DEVICE).eval()

    @torch.no_grad()
    def show_attention(node_key: str, pair_text: str, layer=-1):
        q = NODE_Q[node_key]
        enc = viz_tok(q, pair_text, truncation=True, max_length=MAX_LENGTH,
                      return_tensors='pt', add_special_tokens=True).to(DEVICE)
        out = viz_model(**enc)
        pred_code = int(out.logits.argmax(-1).cpu())

        attn = out.attentions[layer][0]           # [heads, seq, seq]
        cls = attn.mean(dim=0)[0]                 # ヘッド平均 → CLS 行
        tokens = viz_tok.convert_ids_to_tokens(enc['input_ids'][0])
        w = cls.clone(); w[0] = 0.0               # CLS自身は除外
        w = (w / (w.max() + 1e-9)).cpu().tolist()

        html = ''.join(
            f'<span style="background-color: rgba(255,80,80,{a:.3f}); padding:2px; margin:1px; '
            f'border-radius:3px;">{t}</span> ' for t, a in zip(tokens, w))
        display(HTML(f'<b>{node_key} / 予測code={pred_code}</b><br>'
                     f'<span style="color:#888">A={q}</span><br>{html}'))

    # 各プロトコルから1件ずつ
    for _, r in ex.groupby('proto', group_keys=False).head(1).iterrows():
        show_attention(r['node'], r['text'])
else:
    print('スキップ（RUN_ATTENTION_VIZ=False または final_model 未作成）')

# 11. まとめ・運用メモ### このノートがやったこと- `dataset/question_calls/02〜09`（8プロトコル・43ノード・10,200ペア）を**1本のBERT**で学習- 入力は verify05 と同じ**文ペア `(対象ノードの質問文, ペア本文)`**、出力は `0=非該当 / 1..N=選択肢`- 5-fold CV(node×label層化) の OOF 予測で、**症状別 / ノード別 / code別**の正答率を集計### 落ちても平気な仕組み（Colab前提）| 保存物 | パス | タイミング ||---|---|---|| 学習ログ(JSONL) | `CKPT_DIR/logs/train_<時刻>.jsonl` | 50ステップごと＋エポック完了ごとに flush || 学習途中の重み+optimizer | `CKPT_DIR/fold<k>_ckpt.pt` | **1エポックごとに上書き**（tmp→replace） || foldのOOF予測 | `CKPT_DIR/fold<k>_pred.csv` | fold完了時（これがあるとfold丸ごとスキップ） || 全OOF | `CKPT_DIR/oof_all.csv` | CV完了時 || 最終モデル | `CKPT_DIR/final_model/` | セル8完了時 |→ **ランタイムが切れたら、セル1から順に再実行するだけ**。完了済みfoldは `fold_skip` ログを出して即復元、学習途中のfoldは `resume` ログを出して次エポックから再開する。- 全部やり直したい: `RESUME=False`、または `CKPT_DIR/fold*_pred.csv` と `fold*_ckpt.pt` を削除- 各foldの重みも残したい: `KEEP_FOLD_WEIGHTS=True`（≈450MB × 5 = 2.2GB を Drive に置くので注意）- 動作確認だけ: `SAMPLE_PER_NODE=40`, `N_FOLDS=2`, `NUM_EPOCHS=1`### 共通(common)モデルを学習するとき1. 共通質問のペアCSVを `dataset/` 配下に置く（列は `ID / ペア / is_<node> / label_<node>`、`0=非該当`）2. セル2の `GROUP_FILES['common']` にそのグロブを追加3. `GROUP = 'common'` にして実行 → `CKPT_DIR` が `checkpoints/verify06_common/` に分かれるので症状別モデルと共存できる### 注意点- **`ID` にラベルが埋まっている**（例 `dyspnea_asthma_history_1_0001` の `_1_`）。特徴量には使っていないが、  今後 `ID` を入力に混ぜるとリークになる。- ノードキーは**プロトコル番号で名前空間化**している（`08_trauma` と `09_trauma` は別ノード扱い）。  背部痛と腰痛はノード構成が同一なので、統合したい場合は `node` の作り方を変える。- 質問文は**データからの自動抽出**（該当ペアの最頻Dispatcher発話）。`node_info.csv` を見て変なものは  `NODE_Q_OVERRIDE` に足すこと。**final_model/node_questions.json に一緒に保存**しているので推論時も一致する。